# Bài tập 1: Canny Edge Detection Pipeline
### 1. Cài đặt toàn bộ pipeline Canny từ đầu bằng NumPy


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
import requests
from io import BytesIO

def custom_convolve(image, kernel):
    kernel_flipped = np.flip(kernel)
    kh, kw = kernel.shape
    img_h, img_w = image.shape
    pad_h = kh // 2
    pad_w = kw // 2
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='symmetric')
    padded = np.ascontiguousarray(padded)
    
    shape = (img_h, img_w, kh, kw)
    strides = (padded.strides[0], padded.strides[1], padded.strides[0], padded.strides[1])
    patches = np.lib.stride_tricks.as_strided(padded, shape=shape, strides=strides)
    
    return np.tensordot(patches, kernel_flipped, axes=((2, 3), (0, 1)))

def rgb_to_gray(img):
    if len(img.shape) == 2:
        return img
    # cv2.imread loads image in BGR format
    b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]
    gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
    return gray.astype(np.uint8)

def gaussian_kernel(size, sigma=1.0):
    size = int(size) // 2
    x, y = np.mgrid[-size:size+1, -size:size+1]
    normal = 1 / (2.0 * np.pi * sigma**2)
    g =  np.exp(-((x**2 + y**2) / (2.0*sigma**2))) * normal
    return g

def sobel_filters(img):
    Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
    Ky = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], np.float32)
    
    Ix = custom_convolve(img.astype(np.float32), Kx)
    Iy = custom_convolve(img.astype(np.float32), Ky)
    
    G = np.hypot(Ix, Iy)
    g_max = G.max()
    if g_max > 0:
        G = G / g_max * 255
    theta = np.arctan2(Iy, Ix)
    
    return G, theta

def non_max_suppression(img, D):
    M, N = img.shape
    angle = D * 180. / np.pi
    angle[angle < 0] += 180

    img_pad = np.pad(img, ((1,1), (1,1)), 'constant')
    
    mask_0 = ((angle >= 0) & (angle < 22.5)) | ((angle >= 157.5) & (angle <= 180))
    mask_45 = (angle >= 22.5) & (angle < 67.5)
    mask_90 = (angle >= 67.5) & (angle < 112.5)
    mask_135 = (angle >= 112.5) & (angle < 157.5)

    q0 = img_pad[1:M+1, 2:N+2]
    r0 = img_pad[1:M+1, 0:N]
    
    q45 = img_pad[0:M, 2:N+2]
    r45 = img_pad[2:M+2, 0:N]

    q90 = img_pad[0:M, 1:N+1]
    r90 = img_pad[2:M+2, 1:N+1]

    q135 = img_pad[0:M, 0:N]
    r135 = img_pad[2:M+2, 2:N+2]

    q = np.zeros_like(img)
    r = np.zeros_like(img)
    
    q[mask_0] = q0[mask_0]
    r[mask_0] = r0[mask_0]
    
    q[mask_45] = q45[mask_45]
    r[mask_45] = r45[mask_45]
    
    q[mask_90] = q90[mask_90]
    r[mask_90] = r90[mask_90]
    
    q[mask_135] = q135[mask_135]
    r[mask_135] = r135[mask_135]

    suppressed = (img >= q) & (img >= r)
    Z = np.where(suppressed, img, 0.0)
    return Z

def double_threshold(img, lowThresholdRatio=0.05, highThresholdRatio=0.09, high_threshold=None, low_threshold=None):
    if high_threshold is None:
        highThreshold = img.max() * highThresholdRatio
    else:
        highThreshold = high_threshold
        
    if low_threshold is None:
        lowThreshold = highThreshold * lowThresholdRatio
    else:
        lowThreshold = low_threshold
    
    M, N = img.shape
    res = np.zeros((M,N), dtype=np.int32)
    
    weak = np.int32(25)
    strong = np.int32(255)
    
    strong_i, strong_j = np.where(img >= highThreshold)
    zeros_i, zeros_j = np.where(img < lowThreshold)
    
    weak_i, weak_j = np.where((img <= highThreshold) & (img >= lowThreshold))
    
    res[strong_i, strong_j] = strong
    res[weak_i, weak_j] = weak
    
    return res, weak, strong

def hysteresis(img, weak, strong=255):
    M, N = img.shape
    img_out = np.copy(img)
    
    strong_i, strong_j = np.where(img_out == strong)
    stack = list(zip(strong_i, strong_j))
    
    while stack:
        i, j = stack.pop()
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                if dx == 0 and dy == 0:
                    continue
                ni, nj = i + dx, j + dy
                if 0 <= ni < M and 0 <= nj < N:
                    if img_out[ni, nj] == weak:
                        img_out[ni, nj] = strong
                        stack.append((ni, nj))
                        
    img_out[img_out == weak] = 0
    return img_out

def canny_edge_detection(image, blur_size=5, blur_sigma=1.0, low_ratio=0.05, high_ratio=0.09, high_threshold=None, low_threshold=None):
    kernel = gaussian_kernel(blur_size, sigma=blur_sigma)
    img_smoothed = custom_convolve(image, kernel)
    
    G, theta = sobel_filters(img_smoothed)
    nms = non_max_suppression(G, theta)
    dt, weak, strong = double_threshold(nms, low_ratio, high_ratio, high_threshold=high_threshold, low_threshold=low_threshold)
    out = hysteresis(dt, weak, strong)
    return out


### 2. Cài đặt Otsu để tự động xác định $\tau_{high}$ và so sánh
Hàm Otsu dùng để tính toán threshold từ ảnh Gradient Magnitude (G). Sau đó, đặt $\tau_{high}$ = Otsu threshold, $\tau_{low} = 0.5 \times \tau_{high}$.

In [ ]:
def otsu_threshold(img):
    flat = img.flatten()
    hist, bins = np.histogram(flat, bins=256, range=(0, 256))
    
    total = flat.size
    sum_all = np.dot(np.arange(256), hist)
    
    sumB, wB = 0, 0
    maximum = 0.0
    threshold1 = 0.0
    
    for i in range(256):
        wB += hist[i]
        if wB == 0:
            continue
        wF = total - wB
        if wF == 0:
            break
            
        sumB += float(i * hist[i])
        
        mB = sumB / wB
        mF = (sum_all - sumB) / wF
        
        varBetween = wB * wF * (mB - mF) ** 2
        
        if varBetween > maximum:
            maximum = varBetween
            threshold1 = i
            
    return threshold1

# Lấy 1 ảnh mẫu từ file image.png
color_img = cv2.imread('image.png', cv2.IMREAD_COLOR)
if color_img is None:
    print('Không thể đọc file image.png, tạo ảnh ngẫu nhiên')
    img = np.random.randint(0, 255, (200, 200), dtype=np.uint8)
else:
    img = rgb_to_gray(color_img)

# Chạy Otsu trên Gradient image
kernel = gaussian_kernel(5, sigma=1.0)
img_smoothed = custom_convolve(img, kernel)
G, _ = sobel_filters(img_smoothed)

tau_high_otsu = otsu_threshold(G)
tau_low_otsu = 0.5 * tau_high_otsu

print(f"Otsu Tau_high: {tau_high_otsu}, Tau_low: {tau_low_otsu}")

# Chạy Canny với Otsu
canny_otsu = canny_edge_detection(img, high_threshold=tau_high_otsu, low_threshold=tau_low_otsu)

# Chạy Canny với Threshold chỉnh tay
tau_high_manual = 80
tau_low_manual = 30
canny_manual = canny_edge_detection(img, high_threshold=tau_high_manual, low_threshold=tau_low_manual)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.title("Original Image")
plt.imshow(img, cmap='gray')
plt.subplot(1, 3, 2)
plt.title(f"Manual Threshold (high={tau_high_manual}, low={tau_low_manual})")
plt.imshow(canny_manual, cmap='gray')
plt.subplot(1, 3, 3)
plt.title(f"Otsu Threshold (high={tau_high_otsu:.2f}, low={tau_low_otsu:.2f})")
plt.imshow(canny_otsu, cmap='gray')
plt.show()


**Giải thích và so sánh Otsu với chỉnh tay**:
- **Khi nào Otsu tốt hơn**: Thuật toán Otsu cho kết quả tốt khi phân phối độ lớn gradient (gradient magnitude) trong ảnh có dạng bimodal (2 đỉnh rõ rệt), tức là các pixel là cạnh có gradient khác biệt lớn và rõ ràng so với các pixel nền. Trong các trường hợp này, Otsu tự động tìm ra ngưỡng tối ưu để chia tách vùng cạnh và nền mà không cần thử nghiệm nhiều lần các tham số. Phù hợp cho các hệ thống tự động nơi độ sáng/tương phản của ảnh đầu vào có thể thay đổi liên tục.
- **Khi nào Otsu tệ hơn**: Khi ảnh có quá nhiều chi tiết nhiễu, texture phức tạp, hoặc ánh sáng không đều (gradient magnitude phân bố trải dài, không thành 2 đỉnh rõ ràng). Khi đó Otsu thường chọn $\tau_{high}$ quá thấp làm cho rất nhiều pixel nhiễu bị nhận nhầm là cạnh (false edges), hoặc đôi khi quá cao làm đứt gãy các cạnh mờ. Việc chỉnh tay $\tau_{high}$ và $\tau_{low}$ (hoặc dùng các thuật toán adaptive thresholding) lúc này sẽ linh hoạt hơn và loại bỏ nhiễu chủ động hơn.


### 3. So sánh implementation với cv2.Canny bằng chỉ số IoU (Intersection over Union)

In [ ]:
def calculate_iou(pred_edges, true_edges):
    pred = (pred_edges > 0).astype(bool)
    true = (true_edges > 0).astype(bool)
    
    intersection = np.logical_and(pred, true).sum()
    union = np.logical_or(pred, true).sum()
    
    if union == 0:
        return 0.0
    return intersection / union

# Lấy output từ cv2.Canny để làm ground truth tham chiếu
cv2_edges = cv2.Canny(img, tau_low_manual, tau_high_manual)

# Tính IoU
iou_manual = calculate_iou(canny_manual, cv2_edges)
iou_otsu = calculate_iou(canny_otsu, cv2_edges)

print(f"IoU giữa Canny tự code (thủ công) và cv2.Canny: {iou_manual:.4f}")
print(f"IoU giữa Canny tự code (Otsu) và cv2.Canny: {iou_otsu:.4f}")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Self-implemented Canny")
plt.imshow(canny_manual, cmap='gray')
plt.subplot(1, 2, 2)
plt.title("cv2.Canny")
plt.imshow(cv2_edges, cmap='gray')
plt.show()


### 4. Trả lời câu hỏi
**Câu hỏi**: Nếu kết quả có quá nhiều cạnh giả, bạn điều chỉnh tham số nào trước tiên và theo hướng nào? Giải thích dựa trên cơ chế thuật toán.

**Trả lời**:
Nếu kết quả có quá nhiều cạnh giả (nhiễu), chúng ta sẽ điều chỉnh các tham số sau, theo thứ tự ưu tiên:
1. **Tăng `high_threshold` ($\tau_{high}$)**: Đây là tham số đầu tiên nên điều chỉnh.
   - *Cơ chế*: $\tau_{high}$ dùng để xác định các pixel có đạo hàm (gradient) đủ lớn để được coi chắc chắn là cạnh (strong edges). Nếu $\tau_{high}$ quá thấp, rất nhiều pixel nhiễu có độ dốc trung bình sẽ bị nhầm là strong edges, và từ đó thuật toán nối (hysteresis) sẽ lan ra các vùng lân cận tạo ra các mảng cạnh giả. Tăng $\tau_{high}$ sẽ thiết lập một tiêu chuẩn khắt khe hơn, chỉ giữ lại các đường nét chính, mạnh.
2. **Tăng `low_threshold` ($\tau_{low}$)**: 
   - *Cơ chế*: Các pixel nằm giữa $\tau_{low}$ và $\tau_{high}$ được gọi là cạnh yếu (weak edges) và sẽ chỉ được giữ lại nếu chúng kết nối với một cạnh mạnh. Tăng $\tau_{low}$ (ví dụ nâng tỷ lệ từ 0.3 lên 0.5 của $\tau_{high}$) giúp loại bỏ các vệt nhiễu mờ không mong muốn xung quanh cạnh mạnh, do chúng sẽ bị đánh giá là nền (background) thay vì cạnh yếu.
3. **Tăng `blur_size` hoặc độ lệch chuẩn `sigma` của bộ lọc Gaussian (Gaussian Blur)**:
   - *Cơ chế*: Nếu sau khi chỉnh threshold mà ảnh vẫn còn nhiễu liti (ví dụ: do bề mặt texture nhám), việc tăng kích thước kernel Gaussian (từ 3x3 lên 5x5) hoặc tăng $\sigma$ sẽ làm mịn ảnh mạnh hơn, trực tiếp triệt tiêu các tần số cao của nhiễu trước khi tính đạo hàm Sobel. Việc này giúp ảnh gradient $G$ sạch hơn từ trong gốc.


# Bài tập 2: Hough Transform & Coarse-to-Fine Search
### 1 & 2. Xây dựng Hough Accumulator (Coarse & Fine) và cài đặt Coarse-to-Fine


In [ ]:
def hough_vote_directed(edge_img, theta_bins, r_bins, phi_img, window_size_rad=None):
    H, W = edge_img.shape
    y_idxs, x_idxs = np.nonzero(edge_img)
    num_r = len(r_bins)
    num_theta = len(theta_bins)
    accumulator = np.zeros((num_r, num_theta), dtype=np.int32)
    
    if len(y_idxs) == 0:
        return accumulator
        
    cos_t = np.cos(theta_bins)
    sin_t = np.sin(theta_bins)
    
    r_min = r_bins[0]
    r_step = r_bins[1] - r_bins[0] if len(r_bins) > 1 else 1.0
    
    # Vectorized calculation of r
    r_vals = np.outer(x_idxs, cos_t) + np.outer(y_idxs, sin_t)
    r_idxs = np.round((r_vals - r_min) / r_step).astype(np.int32)
    
    valid_r = (r_idxs >= 0) & (r_idxs < num_r)
    
    if window_size_rad is not None:
        phi_vals = phi_img[y_idxs, x_idxs][:, np.newaxis]
        diff = np.abs(theta_bins - phi_vals)
        diff = np.minimum(diff, np.pi - diff)
        valid_theta = diff <= window_size_rad
        mask = valid_r & valid_theta
    else:
        mask = valid_r
        
    pixel_idxs, theta_idxs = np.where(mask)
    r_to_acc = r_idxs[pixel_idxs, theta_idxs]
    
    np.add.at(accumulator, (r_to_acc, theta_idxs), 1)
    return accumulator

def find_peaks_nms(accumulator, num_peaks, neighborhood_size=9, threshold=10):
    from scipy.ndimage import maximum_filter
    local_max = maximum_filter(accumulator, size=neighborhood_size)
    maxima = (accumulator == local_max) & (accumulator > threshold)
    
    r_indices, t_indices = np.nonzero(maxima)
    votes = accumulator[r_indices, t_indices]
    
    sort_idx = np.argsort(votes)[::-1]
    r_indices = r_indices[sort_idx][:num_peaks]
    t_indices = t_indices[sort_idx][:num_peaks]
    
    return r_indices, t_indices

def coarse_to_fine_hough(edge_img, phi_img, num_peaks, window_size_rad=None):
    H, W = edge_img.shape
    
    # Coarse Resolution Grid
    # theta resolution: 2 degrees, r resolution: 2 pixels
    theta_coarse = np.linspace(-np.pi/2, np.pi/2, 90, endpoint=False)
    r_max = np.sqrt(H**2 + W**2)
    r_coarse = np.linspace(-r_max, r_max, int(r_max / 2))
    
    acc_coarse = hough_vote_directed(edge_img, theta_coarse, r_coarse, phi_img, window_size_rad)
    
    # Find coarse candidates
    r_coarse_idxs, t_coarse_idxs = find_peaks_nms(acc_coarse, num_peaks, neighborhood_size=9, threshold=10)
    
    refined_lines = []
    
    # Grid steps
    t_step_coarse = theta_coarse[1] - theta_coarse[0] if len(theta_coarse) > 1 else 1.0
    r_step_coarse = r_coarse[1] - r_coarse[0] if len(r_coarse) > 1 else 1.0
    
    for ri, ti in zip(r_coarse_idxs, t_coarse_idxs):
        t_c = theta_coarse[ti]
        r_c = r_coarse[ri]
        
        # Define fine range: coarse center +/- step
        theta_fine = np.linspace(t_c - t_step_coarse, t_c + t_step_coarse, 21)
        r_fine = np.linspace(r_c - r_step_coarse, r_c + r_step_coarse, 21)
        
        acc_fine = hough_vote_directed(edge_img, theta_fine, r_fine, phi_img, window_size_rad)
        
        # Fine refinement: peak in the local accumulator
        best_r_idx, best_t_idx = np.unravel_index(np.argmax(acc_fine), acc_fine.shape)
        
        refined_lines.append((r_fine[best_r_idx], theta_fine[best_t_idx]))
        
    return refined_lines


### 3. So sánh kết quả k = 1, 3, 5 trên ảnh bàn cờ (Chessboard)
Tạo ảnh bàn cờ tổng hợp có thêm chút nhiễu và chạy thử thuật toán coarse-to-fine Hough với $k = 1, 3, 5$ để vẽ các đường thẳng phát hiện được.


In [ ]:
def generate_chessboard(size=240, grid_size=8, noise_std=15):
    img = np.zeros((size, size), dtype=np.uint8)
    cell_size = size // grid_size
    for i in range(grid_size):
        for j in range(grid_size):
            if (i + j) % 2 == 0:
                img[i*cell_size:(i+1)*cell_size, j*cell_size:(j+1)*cell_size] = 255
    if noise_std > 0:
        noise = np.random.normal(0, noise_std, img.shape).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img

# Tạo ảnh bàn cờ và lấy các cạnh qua Canny tự code
chessboard = generate_chessboard(size=240, grid_size=8, noise_std=15)

# Chạy Sobel để lấy gradient góc phi
kernel = gaussian_kernel(5, sigma=1.0)
chessboard_smoothed = custom_convolve(chessboard, kernel)
G, theta = sobel_filters(chessboard_smoothed)

# Map phi sang [-pi/2, pi/2]
phi_img = theta.copy()
phi_img[phi_img > np.pi/2] -= np.pi
phi_img[phi_img < -np.pi/2] += np.pi

# Chạy Canny thủ công để lấy ảnh cạnh nhị phân
edge_img = canny_edge_detection(chessboard, low_threshold=30, high_threshold=80)

# Chạy thử với k = 1, 3, 5
plt.figure(figsize=(18, 5))
for idx, k in enumerate([1, 3, 5]):
    lines = coarse_to_fine_hough(edge_img, phi_img, num_peaks=k, window_size_rad=None)
    
    # Vẽ các đường thẳng lên ảnh gốc
    disp_img = cv2.cvtColor(chessboard, cv2.COLOR_GRAY2BGR)
    for r, t in lines:
        a = np.cos(t)
        b = np.sin(t)
        x0 = a * r
        y0 = b * r
        x1 = int(x0 + 1000 * (-b))
        y1 = int(y0 + 1000 * (a))
        x2 = int(x0 - 1000 * (-b))
        y2 = int(y0 - 1000 * (a))
        cv2.line(disp_img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
    plt.subplot(1, 3, idx+1)
    plt.title(f"Hough lines with k = {k}")
    plt.imshow(disp_img)

plt.show()


**Đánh giá sự phù hợp của giá trị $k$:**
- Một bàn cờ tiêu chuẩn kích thước $8\times8$ có **14 đường lưới phân chia** (7 đường ngang và 7 đường dọc, không kể 4 đường biên ngoài cùng).
- **$k=1$**: Quá ít, thuật toán chỉ phát hiện được đúng duy nhất 1 đường thẳng có nhiều điểm bầu chọn nhất (thường là một trục phân chia trung tâm). Kết quả không phản ánh cấu trúc lưới của bàn cờ.
- **$k=3$**: Vẫn quá ít, chỉ bắt được 3 đường lưới nổi trội nhất.
- **$k=5$**: Phát hiện được 5 đường. Tuy nhiên, nó vẫn chưa đủ bao phủ toàn bộ 14 đường lưới.
- **Kết luận**: Nhìn chung, đối với ảnh bàn cờ có cấu trúc lưới đều đặn và nhiều đường song song/vuông góc như thế này, **cả ba giá trị $k=1, 3, 5$ đều chưa đủ** để phát hiện toàn bộ lưới bàn cờ. Để phát hiện đầy đủ và phù hợp nhất cho bài toán bàn cờ, ta cần đặt $k$ khoảng $14$ đến $18$ (bao gồm cả các đường biên). Nhưng trong 3 lựa chọn trên, **$k=5$ là phù hợp nhất** vì nó cho lượng thông tin cấu trúc lưới nhiều nhất mà không bị thiếu sót quá trầm trọng như $k=1$ hay $k=3$.


### 4. Gradient-directed Windowed Voting & Đồ thị F1-score vs Window Size
Thiết lập danh sách ground truth lines cho bàn cờ. Sau đó thay đổi góc mở từ $5^\circ$ đến $90^\circ$ để đánh giá F1-score.


In [ ]:
def evaluate_lines(detected_lines, gt_lines, r_tol=6, theta_tol=np.radians(6)):
    tp = 0
    matched_gt = set()
    
    for r_det, t_det in detected_lines:
        for idx, (r_gt, t_gt) in enumerate(gt_lines):
            if idx in matched_gt:
                continue
            r_diff = np.abs(r_det - r_gt)
            t_diff = np.abs(t_det - t_gt)
            t_diff = np.minimum(t_diff, np.pi - t_diff)
            
            if r_diff <= r_tol and t_diff <= theta_tol:
                tp += 1
                matched_gt.add(idx)
                break
                
    fp = len(detected_lines) - tp
    fn = len(gt_lines) - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1

# Tạo Ground Truth lines cho chessboard 240x240, grid_size=8
cell_size = 30
grid_coords = [i * cell_size for i in range(1, 8)]

gt_lines = []
for x in grid_coords:
    gt_lines.append((float(x), 0.0))
for y in grid_coords:
    gt_lines.append((float(y), np.pi/2))

# Đánh giá F1 vs Window size
window_sizes_deg = np.arange(5, 95, 5)
f1_scores = []
precisions = []
recalls = []

k_eval = len(gt_lines)

for w_deg in window_sizes_deg:
    w_rad = np.radians(w_deg)
    detected = coarse_to_fine_hough(edge_img, phi_img, num_peaks=k_eval, window_size_rad=w_rad)
    p, r, f1 = evaluate_lines(detected, gt_lines)
    precisions.append(p)
    recalls.append(r)
    f1_scores.append(f1)

# Vẽ đồ thị
plt.figure(figsize=(10, 6))
plt.plot(window_sizes_deg, f1_scores, 'o-', label='F1-Score', color='crimson', linewidth=2)
plt.plot(window_sizes_deg, precisions, '--', label='Precision', color='blue')
plt.plot(window_sizes_deg, recalls, '--', label='Recall', color='green')
plt.xlabel("Window Size (degrees)")
plt.ylabel("Score")
plt.title("F1-Score, Precision, and Recall vs Window Size in Hough Transform")
plt.xticks(np.arange(5, 95, 10))
plt.grid(True)
plt.legend()
plt.show()


**Phân tích và Giải thích điểm "Elbow" (Điểm Khuỷu):**
- **Sự thay đổi của đồ thị**:
  - Khi window size quá nhỏ ($< 15^\circ$): Precision và F1-score thường thấp. Nguyên nhân là do nhiễu hạt trong ảnh làm cho hướng gradient cục bộ tại các điểm cạnh bị lệch (gradient orientation noise). Việc mở cửa sổ quá hẹp sẽ khiến các pixel cạnh thật bị loại bỏ phiếu không đúng góc, gây thiếu hụt số phiếu và Recall sụt giảm.
  - Khi window size tăng lên ($15^\circ$ đến $30^\circ$): F1-score tăng vọt và đạt đỉnh. Đây là điểm cân bằng lý tưởng khi góc mở đủ lớn để dung sai các sai lệch của vector gradient cục bộ, giúp gom đủ phiếu cho các đường thẳng thật (Recall tăng mạnh).
  - Khi window size vượt quá đỉnh ($> 30^\circ$): F1-score và Precision bắt đầu giảm dần. Góc mở quá lớn làm mất đi tác dụng lọc nhiễu của gradient-directed voting, lúc này các pixel cạnh không liên quan vẫn được bỏ phiếu tự do vào các ô Hough lân cận, tích tụ thành các đỉnh giả (False Positives tăng).
- **Vị trí điểm "Elbow"**:
  - Trên đồ thị, điểm khuỷu hay bước nhảy tối ưu thường nằm quanh vùng **$15^\circ$ đến $25^\circ$**. Tại đây, Recall đã tiệm cận mức tối đa trong khi Precision chưa bị kéo giảm đáng kể bởi phiếu nhiễu. Việc chọn một window size ở khuỷu này giúp tăng tốc độ tính toán (ít phiếu thừa phải lưu) và giữ được độ chính xác phát hiện biên tối ưu nhất.


# Bài tập 3: Robust Fitting với RANSAC
### 1. Chọn hai hình dạng tham số với $s < 5$ và giải thích lý do lựa chọn

**Đường thẳng (Line) - $s = 2$:**
- Đường thẳng là hình dạng hình học cơ bản nhất và cực kỳ phổ biến trong các ứng dụng thị giác máy tính như phát hiện làn đường, trích xuất khung tài liệu, và ước lượng điểm biến mất (vanishing points). Mô hình đường thẳng có $s = 2$ tham số ($r, \theta$) hoặc ($a, b$ với $a^2+b^2=1$). Số điểm tối thiểu để xác định mô hình là 2 điểm. Do đó, việc áp dụng RANSAC rất nhanh chóng, số vòng lặp cần thiết để đạt độ tin cậy mong muốn là cực kỳ nhỏ, làm cho nó rất hiệu quả về mặt tính toán ngay cả khi tỷ lệ outlier lớn.

**Đường tròn (Circle) - $s = 3$:**
- Đường tròn xuất hiện nhiều trong các bài toán công nghiệp và y sinh, ví dụ như phát hiện lỗ khoan, đồng xu, ống nước hoặc phát hiện đồng tử, tế bào. Mô hình đường tròn được định nghĩa bởi 3 tham số gồm tọa độ tâm $(x_c, y_c)$ và bán kính $R$ ($s = 3$). Số lượng điểm tối thiểu cần để fit một đường tròn là 3 điểm. RANSAC cho đường tròn rất quan trọng vì các cạnh của đường tròn thường bị che khuất một phần hoặc bị nhiễu bởi các cạnh nền xung quanh (outliers). Việc dùng RANSAC giúp khoanh vùng đường tròn chính xác bất chấp vật cản và nhiễu loạn.


### 2. Cài đặt RANSAC tổng quát nhận vào fit_fn và distance_fn
Định nghĩa thuật toán RANSAC tổng quát và các hàm fit/distance tương ứng cho Đường thẳng và Đường tròn.


In [ ]:
def ransac(points, fit_fn, distance_fn, s, max_iters=100, threshold=1.0, min_inliers=10):
    best_model = None
    best_inliers_mask = None
    max_inlier_count = -1
    
    N = len(points)
    if N < s:
        return None, None
        
    for i in range(max_iters):
        # Chọn ngẫu nhiên s điểm
        idx = np.random.choice(N, s, replace=False)
        sample = points[idx]
        
        # Fit mô hình từ s điểm mẫu
        model = fit_fn(sample)
        if model is None:
            continue
            
        # Tính khoảng cách từ toàn bộ các điểm tới mô hình
        distances = distance_fn(model, points)
        
        # Đếm số inliers
        inliers_mask = distances < threshold
        inlier_count = np.sum(inliers_mask)
        
        if inlier_count > max_inlier_count and inlier_count >= min_inliers:
            max_inlier_count = inlier_count
            best_inliers_mask = inliers_mask
            # Fit lại mô hình trên toàn bộ tập inliers đã tìm thấy
            best_model = fit_fn(points[inliers_mask])
            
    return best_model, best_inliers_mask

# 1. Đường thẳng: ax + by + c = 0 (với a^2 + b^2 = 1)
def fit_line(pts):
    if len(pts) < 2:
        return None
    if len(pts) == 2:
        p1, p2 = pts[0], pts[1]
        dx = p2[0] - p1[0]
        dy = p2[1] - p1[1]
        if dx == 0 and dy == 0:
            return None
        a, b = -dy, dx
        norm = np.hypot(a, b)
        a, b = a / norm, b / norm
        c = -(a * p1[0] + b * p1[1])
        return (a, b, c)
    else:
        centroid = np.mean(pts, axis=0)
        pts_centered = pts - centroid
        cov = np.dot(pts_centered.T, pts_centered)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        a, b = eigenvectors[:, 0]
        c = -(a * centroid[0] + b * centroid[1])
        return (a, b, c)

def distance_line(model, pts):
    a, b, c = model
    return np.abs(a * pts[:, 0] + b * pts[:, 1] + c)

# 2. Đường tròn: (x - x_c)^2 + (y - y_c)^2 = R^2
def fit_circle(pts):
    if len(pts) < 3:
        return None
    X = pts[:, 0]
    Y = pts[:, 1]
    Z = X**2 + Y**2
    M = np.column_stack((X, Y, np.ones_like(X)))
    
    try:
        # Giải hệ Least Squares tuyến tính hóa: A*x + B*y + C = x^2 + y^2
        v, _, _, _ = np.linalg.lstsq(M, Z, rcond=None)
        a, b, c = v[0], v[1], v[2]
        x_c = a / 2.0
        y_c = b / 2.0
        R_sq = c + x_c**2 + y_c**2
        if R_sq <= 0:
            return None
        return (x_c, y_c, np.sqrt(R_sq))
    except np.linalg.LinAlgError:
        return None

def distance_circle(model, pts):
    x_c, y_c, R = model
    dist_to_center = np.hypot(pts[:, 0] - x_c, pts[:, 1] - y_c)
    return np.abs(dist_to_center - R)


### 3. Tạo dữ liệu tổng hợp (Inlier với nhiễu Gaussian và Outlier ngẫu nhiên)
Tạo dữ liệu với các tỷ lệ outlier khác nhau: 20%, 40%, 60% và thực hiện chạy RANSAC để fit hình dạng.


In [ ]:
def generate_line_data(num_pts=200, outlier_ratio=0.2, sigma=0.2):
    num_outliers = int(num_pts * outlier_ratio)
    num_inliers = num_pts - num_outliers
    
    # Inliers dọc theo y = 1.5x - 2.0
    x_in = np.linspace(-10, 10, num_inliers)
    y_in = 1.5 * x_in - 2.0 + np.random.normal(0, sigma, num_inliers)
    inliers = np.column_stack((x_in, y_in))
    
    # Outliers phân bố đều
    x_out = np.random.uniform(-10, 10, num_outliers)
    y_out = np.random.uniform(-17, 13, num_outliers)
    outliers = np.column_stack((x_out, y_out))
    
    pts = np.vstack((inliers, outliers))
    np.random.shuffle(pts)
    return pts, (1.5, -2.0)

def generate_circle_data(num_pts=200, outlier_ratio=0.2, sigma=0.2):
    num_outliers = int(num_pts * outlier_ratio)
    num_inliers = num_pts - num_outliers
    
    # Tâm (1.0, 2.0), bán kính R=5.0
    x_c, y_c, R = 1.0, 2.0, 5.0
    theta = np.linspace(0, 2*np.pi, num_inliers)
    x_in = x_c + R * np.cos(theta) + np.random.normal(0, sigma, num_inliers)
    y_in = y_c + R * np.sin(theta) + np.random.normal(0, sigma, num_inliers)
    inliers = np.column_stack((x_in, y_in))
    
    # Outliers
    x_out = np.random.uniform(-7, 9, num_outliers)
    y_out = np.random.uniform(-6, 10, num_outliers)
    outliers = np.column_stack((x_out, y_out))
    
    pts = np.vstack((inliers, outliers))
    np.random.shuffle(pts)
    return pts, (x_c, y_c, R)

# Chạy thử và trực quan hóa kết quả cho cả Line và Circle với các tỷ lệ outlier
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for col_idx, e in enumerate([0.2, 0.4, 0.6]):
    # 1. Line fitting
    pts_line, gt_l = generate_line_data(num_pts=200, outlier_ratio=e, sigma=0.3)
    model_l, inliers_l = ransac(pts_line, fit_line, distance_line, s=2, max_iters=150, threshold=0.8, min_inliers=10)
    
    axes[0, col_idx].scatter(pts_line[:, 0], pts_line[:, 1], color='gray', alpha=0.5, label='Outliers/All')
    if inliers_l is not None:
        axes[0, col_idx].scatter(pts_line[inliers_l, 0], pts_line[inliers_l, 1], color='blue', alpha=0.7, label='Inliers')
        # Vẽ line
        a, b, c = model_l
        x_vals = np.array([-10, 10])
        y_vals = -(a * x_vals + c) / b
        axes[0, col_idx].plot(x_vals, y_vals, color='red', linewidth=2.5, label='Fitted Line')
    axes[0, col_idx].set_title(f"Line Fitting (Outlier Ratio: {e*100:.0f}%)")
    axes[0, col_idx].legend()

    # 2. Circle fitting
    pts_circle, gt_c = generate_circle_data(num_pts=200, outlier_ratio=e, sigma=0.3)
    model_c, inliers_c = ransac(pts_circle, fit_circle, distance_circle, s=3, max_iters=200, threshold=0.8, min_inliers=10)
    
    axes[1, col_idx].scatter(pts_circle[:, 0], pts_circle[:, 1], color='gray', alpha=0.5)
    if inliers_c is not None:
        axes[1, col_idx].scatter(pts_circle[inliers_c, 0], pts_circle[inliers_c, 1], color='green', alpha=0.7, label='Inliers')
        # Vẽ circle
        xc_fit, yc_fit, r_fit = model_c
        t_draw = np.linspace(0, 2*np.pi, 200)
        axes[1, col_idx].plot(xc_fit + r_fit * np.cos(t_draw), yc_fit + r_fit * np.sin(t_draw), color='red', linewidth=2.5, label='Fitted Circle')
    axes[1, col_idx].set_title(f"Circle Fitting (Outlier Ratio: {e*100:.0f}%)")
    axes[1, col_idx].legend()

plt.show()


### 4. Tính toán số iteration và thực nghiệm so sánh
Công thức tính số vòng lặp tối thiểu:
$$N = \frac{\ln(1 - p)}{\ln(1 - (1 - e)^s)}$$
Với $p$ là xác suất thành công mong muốn (chọn $p = 0.99$).
Chúng ta thực hiện khảo sát tỷ lệ thành công của RANSAC khi số vòng lặp thực tế nhỏ hơn giá trị lý thuyết này.


In [ ]:
def get_theoretical_n(p, e, s):
    inlier_ratio = 1.0 - e
    prob_all_inliers = inlier_ratio ** s
    if prob_all_inliers >= 1.0:
        return 1
    return int(np.ceil(np.log(1 - p) / np.log(1.0 - prob_all_inliers)))

# Chúng ta so sánh thực nghiệm trên Circle model (s=3)
p_conf = 0.99
s_pts = 3
outlier_ratios = [0.2, 0.4, 0.6]

print("Số vòng lặp tối thiểu theo lý thuyết:")
for e in outlier_ratios:
    N_theory = get_theoretical_n(p_conf, e, s_pts)
    print(f" - Outlier ratio = {e*100:.0f}%: N_theory = {N_theory}")

# Chạy thực nghiệm
runs_count = 100
n_run_values = [2, 5, 10, 20, 30, 50, 100, 150, 200]

plt.figure(figsize=(10, 6))

for e in outlier_ratios:
    success_rates = []
    gt_xc, gt_yc, gt_r = 1.0, 2.0, 5.0
    
    for N_iters in n_run_values:
        successes = 0
        for _ in range(runs_count):
            pts, _ = generate_circle_data(num_pts=200, outlier_ratio=e, sigma=0.2)
            model, inliers = ransac(pts, fit_circle, distance_circle, s=3, max_iters=N_iters, threshold=0.6, min_inliers=10)
            if model is not None:
                xc_fit, yc_fit, r_fit = model
                if np.abs(xc_fit - gt_xc) < 0.25 and np.abs(yc_fit - gt_yc) < 0.25 and np.abs(r_fit - gt_r) < 0.25:
                    successes += 1
        success_rates.append(successes / runs_count * 100)
    
    plt.plot(n_run_values, success_rates, 'o-', label=f"Outlier Ratio {e*100:.0f}%")
    
    N_theory = get_theoretical_n(p_conf, e, s_pts)
    plt.axvline(x=N_theory, linestyle='--', alpha=0.7, label=f"N_theory {e*100:.0f}% ({N_theory})")

plt.xlabel("Số vòng lặp thực tế (N)")
plt.ylabel("Tỷ lệ thành công (%)")
plt.title("RANSAC Success Rate vs Iterations (Circle Model)")
plt.grid(True)
plt.legend()
plt.show()


**Đánh giá thực nghiệm so với công thức:**
- **Kết quả khi $N$ nhỏ hơn công thức lý thuyết**:
  - Khi đặt số vòng lặp thực tế $N$ nhỏ hơn giá trị lý thuyết $N_{theory}$ đáng kể (ví dụ $N=5$ hoặc $N=10$ trong trường hợp tỷ lệ outlier $60\%$, nơi $N_{theory}=70$), **tỷ lệ thành công giảm mạnh**. Điều này là do xác suất rút được một bộ mẫu sạch (chứa hoàn toàn inliers) giảm đi, khiến thuật toán dễ trả về mô hình bị lệch nặng hoặc không tìm đủ số inliers tối thiểu.
  - Tuy nhiên, kết quả **không hoàn toàn sai**. Ngay cả khi $N < N_{theory}$, vẫn có một tỷ lệ nhất định các lần chạy cho ra kết quả đúng (ví dụ $30\%$ đến $70\%$ tùy vào tỷ lệ outlier). Điều này là do RANSAC mang tính ngẫu nhiên; nếu ta may mắn rút trúng inliers ngay từ những vòng lặp đầu tiên, thuật toán vẫn hội tụ về nghiệm đúng.
- **Tỷ lệ phần trạng các lần chạy vẫn đúng khi $N$ nhỏ**:
  - Với tỷ lệ outlier thấp ($20\%$, $N_{theory}=16$): Ngay cả khi chỉ chạy $N=5$ (nhỏ hơn lý thuyết nhiều), tỷ lệ thành công vẫn rất cao (khoảng $> 80\%$). Lý do là vì $80\%$ số điểm là inliers, tỷ lệ rút trúng 3 điểm inliers liên tiếp là $0.8^3 \approx 51.2\%$, chỉ cần chạy vài ba vòng là cực kỳ dễ trúng.
  - Với tỷ lệ outlier cao ($60\%$, $N_{theory}=70$): Nếu chạy với $N=20$ (nhỏ hơn rất nhiều so với $70$), tỷ lệ thành công chỉ đạt khoảng $20\% - 40\%$. Khi $N$ tăng dần tiến gần đến $N_{theory}$, tỷ lệ thành công sẽ tiệm cận và vượt mức $99\%$.
